# Lesson 3: Sentence Window Retrieval

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import utils

import os

In [4]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_files=["./eBook-How-to-Build-a-Career-in-AI.pdf"]
).load_data()

In [5]:
print(type(documents), "\n")
print(len(documents), "\n")
print(type(documents[0]))
print(documents[0])

<class 'list'> 

41 

<class 'llama_index.core.schema.Document'>
Doc ID: 9eaafed1-68f7-4b1d-a9c6-b7ff718bc1b4
Text: PAGE 1 Founder, DeepLearning.AI Collected Insights from Andrew
Ng How to  Build Your Career in AI A Simple Guide


In [7]:
from llama_index.core import Document

document = Document(text="\n\n".join([doc.text for doc in documents]))

## Window-sentence retrieval setup

In [8]:
from llama_index.core.node_parser import SentenceWindowNodeParser

# create the sentence window node parser w/ default settings
node_parser = SentenceWindowNodeParser.from_defaults(
    window_size=3,
    window_metadata_key="window",
    original_text_metadata_key="original_text",
)

In [9]:
text = "hello. how are you? I am fine!  "

nodes = node_parser.get_nodes_from_documents([Document(text=text)])

In [10]:
print([x.text for x in nodes])

['hello. ', 'how are you? ', 'I am fine!  ']


In [11]:
print(nodes[1].metadata["window"])

hello.  how are you?  I am fine!  


In [12]:
text = "hello. foo bar. cat dog. mouse"

nodes = node_parser.get_nodes_from_documents([Document(text=text)])

In [13]:
print([x.text for x in nodes])

['hello. ', 'foo bar. ', 'cat dog. ', 'mouse']


In [14]:
print(nodes[0].metadata["window"])

hello.  foo bar.  cat dog.  mouse


### Building the index

In [16]:
from llama_index.llms.google_genai import GoogleGenAI

llm = GoogleGenAI(
    model="gemma-4-31b-it",
    api_key=utils.get_google_api_key(),
    temperature=0.1,
)

2026-07-20 22:34:08,858 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it "HTTP/1.1 200 OK"


In [17]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = llm
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.node_parser = node_parser

2026-07-20 22:34:14,789 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:34:14,791 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-20 22:34:14,814 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-07-20 22:34:15,134 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:34:15,169 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-20 22:34:15,172 - INFO - Loading SentenceTransformer model from BAAI/b

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-20 22:34:17,818 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:34:18,156 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:34:18,459 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:34:18,766 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:34:19,189 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:34:19,212 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json 

In [18]:
from llama_index.core import VectorStoreIndex

sentence_index = VectorStoreIndex.from_documents([document])

In [19]:
sentence_index.storage_context.persist(persist_dir="./sentence_index")


In [20]:
# This block of code is optional to check
# if an index file exist, then it will load it
# if not, it will rebuild it

import os
from llama_index.core import VectorStoreIndex, StorageContext, load_index_from_storage

if not os.path.exists("./sentence_index"):
    sentence_index = VectorStoreIndex.from_documents([document])

    sentence_index.storage_context.persist(persist_dir="./sentence_index")
else:
    sentence_index = load_index_from_storage(
        StorageContext.from_defaults(persist_dir="./sentence_index")
    )

2026-07-20 22:34:25,640 - INFO - Loading all indices.


### Building the postprocessor

In [21]:
from llama_index.core.postprocessor import MetadataReplacementPostProcessor

postproc = MetadataReplacementPostProcessor(
    target_metadata_key="window"
)

In [23]:
from llama_index.core.schema import NodeWithScore
from copy import deepcopy

scored_nodes = [NodeWithScore(node=x, score=1.0) for x in nodes]
nodes_old = [deepcopy(n) for n in nodes]

In [24]:
nodes_old[1].text

'foo bar. '

In [25]:
replaced_nodes = postproc.postprocess_nodes(scored_nodes)

In [26]:
print(replaced_nodes[1].text)

hello.  foo bar.  cat dog.  mouse


### Adding a reranker

In [27]:
from llama_index.core.postprocessor import SentenceTransformerRerank

# BAAI/bge-reranker-base
# link: https://huggingface.co/BAAI/bge-reranker-base
rerank = SentenceTransformerRerank(
    top_n=2, model="BAAI/bge-reranker-base"
)

2026-07-20 22:35:25,358 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-07-20 22:35:25,364 - INFO - No modules.json found for BAAI/bge-reranker-base, initializing a new CrossEncoder model.
2026-07-20 22:35:25,642 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-base "HTTP/1.1 200 OK"
2026-07-20 22:35:25,928 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:35:25,952 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/config.json "HTTP/1.1 200 OK"
2026-07-20 22:35:26,298 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:35:26,572 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-bas

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-07-20 22:35:26,959 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:35:27,230 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:35:27,499 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:35:27,792 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:35:28,061 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:35:28,083 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/tokenizer_config.json 

In [29]:
from llama_index.core import QueryBundle
from llama_index.core.schema import TextNode, NodeWithScore

query = QueryBundle("I want a dog.")

scored_nodes = [
    NodeWithScore(node=TextNode(text="This is a cat"), score=0.6),
    NodeWithScore(node=TextNode(text="This is a dog"), score=0.4),
]

In [30]:
reranked_nodes = rerank.postprocess_nodes(
    scored_nodes, query_bundle=query
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [31]:
print([(x.text, x.score) for x in reranked_nodes])

[('This is a dog', np.float32(0.91827416)), ('This is a cat', np.float32(0.00140408))]


### Runing the query engine

In [32]:
sentence_window_engine = sentence_index.as_query_engine(
    similarity_top_k=6, node_postprocessors=[postproc, rerank]
)

In [33]:
window_response = sentence_window_engine.query(
    "What are the keys to building a career in AI?"
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-20 22:36:01,370 - INFO - AFC is enabled with max remote calls: 10.
2026-07-20 22:36:05,880 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 503 Service Unavailable"
2026-07-20 22:36:05,883 - WARNING - Retrying llama_index.llms.google_genai.base.GoogleGenAI._chat in 1 seconds as it raised ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}.
2026-07-20 22:36:06,889 - ERROR - Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x103dafac0> is already entered
2026-07-20 22:36:06,

In [37]:
from llama_index.core.response.notebook_utils import display_response

display_response(window_response)

2026-07-20 22:38:28,813 - WARNING - Matplotlib is building the font cache; this may take a moment.
2026-07-20 22:38:31,758 - INFO - Failed to extract font properties from /System/Library/PrivateFrameworks/FontServices.framework/Resources/Reserved/PingFangUI.ttc: FT_Open_Face (ft2font.cpp line 200) failed with error 0x02: unknown file format
2026-07-20 22:38:31,786 - INFO - Failed to extract font properties from /System/Library/Fonts/Supplemental/NISC18030.ttf: Non-scalable fonts are not supported
2026-07-20 22:38:31,895 - INFO - Failed to extract font properties from /System/Library/Fonts/LastResort.otf: tuple indices must be integers or slices, not str
2026-07-20 22:38:31,978 - INFO - Failed to extract font properties from /System/Library/Fonts/Apple Color Emoji.ttc: Non-scalable fonts are not supported
2026-07-20 22:38:32,013 - INFO - generated new fontManager


**`Final Response:`** The keys to building a career in AI include three primary steps: learning foundational technical skills, working on projects, and finding a job. These steps are supported by being part of a community. Additionally, the ability to collaborate with, influence, and be influenced by others is critical when working in teams on large projects.

## Putting it all Together

In [38]:
from utils import build_sentence_window_index, get_sentence_window_query_engine

In [47]:
from llama_index.llms.google_genai import GoogleGenAI

index = build_sentence_window_index(
    [document],
    llm=GoogleGenAI(
        model="gemma-4-31b-it",
        api_key=utils.get_google_api_key(),
        temperature=0.1,
    ),
    save_dir="./sentence_index",
)


2026-07-20 22:39:48,972 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it "HTTP/1.1 200 OK"
2026-07-20 22:39:49,396 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:39:49,421 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-07-20 22:39:49,848 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:39:49,869 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-20 22:39:49,872 - INFO - Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
2026-07-20 22:39

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-20 22:39:52,389 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:39:52,696 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:39:52,991 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:39:53,323 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:39:53,690 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:39:53,710 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json 

In [48]:
query_engine = get_sentence_window_query_engine(index, similarity_top_k=6)


2026-07-20 22:39:57,115 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-07-20 22:39:57,120 - INFO - No modules.json found for BAAI/bge-reranker-base, initializing a new CrossEncoder model.
2026-07-20 22:39:57,407 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-base "HTTP/1.1 200 OK"
2026-07-20 22:39:57,698 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:39:57,723 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/config.json "HTTP/1.1 200 OK"
2026-07-20 22:39:58,002 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:39:58,275 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-bas

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-07-20 22:39:58,679 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:39:58,948 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:39:59,233 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:39:59,516 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:39:59,798 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:39:59,820 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/tokenizer_config.json 

## Evaluation (LlamaIndex RAG triad)


In [49]:
eval_questions = []
with open('generated_questions.text', 'r') as file:
    for line in file:
        # Remove newline character and convert to integer
        item = line.strip()
        eval_questions.append(item)

In [50]:
from utils import get_eval_llm, evaluate_query_engine, summarize_eval_df

eval_llm = get_eval_llm()


2026-07-20 22:40:03,906 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it "HTTP/1.1 200 OK"


In [51]:
# Load questions once for comparison runs
# (eval_questions should already be loaded above; this cell is a no-op safeguard)
assert len(eval_questions) > 0


### Sentence window size = 1

In [55]:
sentence_index_1 = build_sentence_window_index(
    documents,
    llm=GoogleGenAI(
        model="gemma-4-31b-it",
        api_key=utils.get_google_api_key(),
        temperature=0.1,
    ),
    embed_model="local:BAAI/bge-small-en-v1.5",
    sentence_window_size=1,
    save_dir="sentence_index_1",
)

2026-07-20 22:40:39,879 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it "HTTP/1.1 200 OK"
2026-07-20 22:40:40,250 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:40:40,274 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-07-20 22:40:40,624 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:40:40,647 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-20 22:40:40,649 - INFO - Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
2026-07-20 22:40

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-20 22:40:42,880 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:40:43,155 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:40:43,496 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:40:43,802 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:40:44,134 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:40:44,157 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json 

In [56]:
sentence_window_engine_1 = get_sentence_window_query_engine(
    sentence_index_1
)

2026-07-20 22:40:47,589 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-07-20 22:40:47,591 - INFO - No modules.json found for BAAI/bge-reranker-base, initializing a new CrossEncoder model.
2026-07-20 22:40:47,871 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-base "HTTP/1.1 200 OK"
2026-07-20 22:40:48,215 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:40:48,237 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/config.json "HTTP/1.1 200 OK"
2026-07-20 22:40:48,524 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:40:48,819 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-bas

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-07-20 22:40:49,265 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:40:49,634 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:40:49,938 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:40:50,214 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-20 22:40:50,666 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-20 22:40:50,688 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/tokenizer_config.json 

In [57]:
records_w1 = evaluate_query_engine(
    sentence_window_engine_1,
    eval_questions,
    app_id="sentence_window_1",
    llm=eval_llm,
)
records_w1.head()


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-20 22:40:57,465 - INFO - AFC is enabled with max remote calls: 10.
2026-07-20 22:41:21,181 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"
2026-07-20 22:41:21,210 - INFO - AFC is enabled with max remote calls: 10.
2026-07-20 22:41:36,268 - INFO - AFC is enabled with max remote calls: 10.
2026-07-20 22:41:59,632 - INFO - AFC is enabled with max remote calls: 10.


,app_id,question,answer,answer_relevancy,context_relevancy,groundedness
0,sentence_window_1,In the context of project selection and execut...,"The ""Ready, Aim, Fire"" and ""Ready, Fire, Aim"" ...",1.0,1.0,1.0


In [58]:
summarize_eval_df(records_w1)


,app_id,answer_relevancy,context_relevancy,groundedness
0,sentence_window_1,1.0,1.0,1.0


In [59]:
print("Window size = 1 evaluation complete.")


Window size = 1 evaluation complete.


### Note about the dataset of questions
- Since this evaluation process takes a long time to run, the following file `generated_questions.text` contains one question (the one mentioned in the lecture video).
- If you would like to explore other possible questions, feel free to explore the file directory by clicking on the "Jupyter" logo at the top right of this notebook. You'll see the following `.text` files:

> - `generated_questions_01_05.text`
> - `generated_questions_06_10.text`
> - `generated_questions_11_15.text`
> - `generated_questions_16_20.text`
> - `generated_questions_21_24.text`

Note that running an evaluation on more than one question can take some time, so we recommend choosing one of these files (with 5 questions each) to run and explore the results.

- For evaluating a personal project, an eval set of 20 is reasonable.
- For evaluating business applications, you may need a set of 100+ in order to cover all the use cases thoroughly.
- Note that since API calls can sometimes fail, you may occasionally see null responses, and would want to re-run your evaluations.  So running your evaluations in smaller batches can also help you save time and cost by only re-running the evaluation on the batches with issues.

In [ ]:
eval_questions = []
with open('generated_questions.text', 'r') as file:
    for line in file:
        # Remove newline character and convert to integer
        item = line.strip()
        eval_questions.append(item)

### Sentence window size = 3

In [ ]:
sentence_index_3 = build_sentence_window_index(
    documents,
    llm=GoogleGenAI(
        model="gemini-2.5-flash",
        api_key=utils.get_google_api_key(),
        temperature=0.1,
    ),
    embed_model="local:BAAI/bge-small-en-v1.5",
    sentence_window_size=3,
    save_dir="sentence_index_3",
)
sentence_window_engine_3 = get_sentence_window_query_engine(
    sentence_index_3
)


In [ ]:
records_w3 = evaluate_query_engine(
    sentence_window_engine_3,
    eval_questions,
    app_id="sentence_window_3",
    llm=eval_llm,
)
summarize_eval_df(records_w3)


In [ ]:
import pandas as pd

comparison = pd.concat(
    [summarize_eval_df(records_w1), summarize_eval_df(records_w3)],
    ignore_index=True,
)
comparison
